# Demo: 2D image RI2FL

> 2D RI2FL demo


In [ ]:
#| default_exp tutorial_2

In [ ]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import aics_pipeline, manifest2csv

from monai.utils import set_determinism

set_determinism(0)

In [ ]:
device = get_device()
print(device)

cuda


### Download Data

In [ ]:
image_path = "../_data/aics"
image_target_paths, data_manifest = aics_pipeline(6, image_path)

Loading manifest: 100%|██████████| 77165/77165 [00:01<00:00, 45.9k/s]


In [ ]:
data_manifest.to_csv(image_path + 'aics_dataset.csv')
manifest2csv(image_target_paths, data_manifest, "ChannelNumberBrightfield","ChannelNumber405", data_save_path=image_path+'/')

### Create Dataloader

In [ ]:
bs, size = 2, 16

itemTfms = [ScaleIntensity(min=0.0, max=1.0),RandCrop2D(size), RandRot90(prob=0.5), RandFlip(prob=0.75)]

data = BioDataLoaders.from_csv(
    image_path, 
    csv_fname='train.csv', 
    header='infer',
    pref='',
    item_tfms=None, # itemTfms
    batch_tfms=None,
    img_cls=BioImageStack,
    target_img_cls=BioImageStack,
    show_summary=True,
    bs=bs,
    )

# print length of training and validation datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

{'blocks': (<fastai.data.block.TransformBlock object>, <fastai.data.block.TransformBlock object>), 'get_items': None, 'get_x': ColReader -- {'cols': 0, 'pref': '', 'suff': '', 'label_delim': None}:
encodes: decodes: , 'get_y': ColReader -- {'cols': 1, 'pref': '/', 'suff': '', 'label_delim': None}:
encodes: decodes: , 'item_tfms': None, 'batch_tfms': None}
{'splitter': <function RandomSplitter.<locals>._inner>, 'path': '../_data/aics', 'bs': 2}


Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


FileNotFoundError: File not found: "/6"

In [ ]:
import pandas as pd
from fastai.data.all import ColReader, DataBlock, DataLoaders
from fastai.vision.all import ImageDataLoaders, ImageBlock

df = pd.DataFrame({'a': '../_data/aics/6677e50c_3500001004_100X_20170623_5-Scene-1-P24-E06.ome.tiff', 'b': ['1']})

# dblock = DataBlock(blocks=(ImageBlock(), ImageBlock),
#                            get_x=ColReader(0),
#                            get_y=ColReader(1),
#                            )
# d = DataLoaders.from_dblock(dblock, df)
d = ImageDataLoaders.from_df(df)

# f = ColReader(['a', 'b'], pref='0', suff='1')

# for o in df.itertuples():
#     print(f(o)[0] + f(o)[1])

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


In [ ]:
data.show_batch(max_n=2, cmap='hot')

### Load and train a 2D model

In [ ]:
from bioMONAI.nets import Deeplab, DeeplabConfig

In [ ]:
config_2d = DeeplabConfig(
    dimensions=2,
    in_channels=1,  
    out_channels=1,
    backbone="resnet10",  
    aspp_dilations=[1, 6, 12, 18]
)
model = Deeplab(config_2d)
 
loss = MSSSIML1Loss(2, levels=2) #CombinedLoss(alpha=0, beta=0.5)
metrics = [SSIMMetric, MSELoss]

trainer = fastTrainer(data, model, loss_fn=loss, metrics=metrics, show_summary=False)

In [ ]:
trainer.fit_flat_cos(500)

In [ ]:
trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!